In [46]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from ipywidgets import VBox, HBox, FloatSlider, Output, FloatLogSlider, Button
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.interpolate import RegularGridInterpolator
import json

%matplotlib inline

In [47]:


def make_array_file_widgets(get_array, set_array, default_path="default_array.json"):
    """
    Create widgets for saving/loading a 2D Python array.

    Parameters
    ----------
    get_array : callable
        Function returning the current 2D array.
    set_array : callable
        Function accepting a new 2D array and updating your editor/display.
    default_path : str
        Default filename/path to save to or load from.
    """

    path_box = widgets.Text(
        value=default_path,
        description="File:",
        layout=widgets.Layout(width="400px")
    )

    save_button = widgets.Button(
        description="Save array",
        button_style="success",
        icon="save"
    )

    load_button = widgets.Button(
        description="Load array",
        button_style="info",
        icon="folder-open"
    )

    status = widgets.Output()

    def save_array(_):
        with status:
            status.clear_output()
            try:
                array = get_array()
                with open(path_box.value, "w", encoding="utf-8") as f:
                    json.dump(array, f, indent=2)
                print(f"Saved array to {path_box.value}")
            except Exception as e:
                print(f"Error saving array: {e}")

    def load_array(_):
        with status:
            status.clear_output()
            try:
                with open(path_box.value, "r", encoding="utf-8") as f:
                    array = json.load(f)

                # Optional validation that it is 2D-list-like
                if not isinstance(array, list) or not all(isinstance(row, list) for row in array):
                    raise ValueError("Loaded file does not contain a 2D list.")

                set_array(array)
                print(f"Loaded array from {path_box.value}")
            except Exception as e:
                print(f"Error loading array: {e}")

    save_button.on_click(save_array)
    load_button.on_click(load_array)

    return widgets.VBox([
        widgets.HBox([path_box, save_button, load_button]),
        status
    ])

In [ ]:
c = 344
n = 128
fig_width = 13
fov = np.pi / 2
todeg = 1 / np.pi * 180

polar_pts = np.meshgrid(
    np.linspace(-np.pi, np.pi, 7 * n),
    np.linspace(0, fov * todeg, n))
cart_pts = np.stack((
    np.sin(polar_pts[0]) * polar_pts[1] / todeg,
    np.cos(polar_pts[0]) * polar_pts[1] / todeg), axis = -1)

ffov = fov + 1e-5
x = np.linspace(-ffov, ffov, n)
y = np.linspace(-ffov, ffov, n)

plot_points = (polar_pts, cart_pts, (x, y))

w_f = FloatLogSlider(
    value=4000, base = 10, min=2, max=np.log10(6000),step=0.001,
    description="Frequency", continuous_update=True, readout_format=".1f"
)
w_idx = FloatSlider(
    value=0, min=0, max=16, step=1,
    description="Index", continuous_update=True, readout_format=".0f"
)
w_y = FloatSlider(
    value=0, min=0, max=0.32, step=0.001,
    description="Y", continuous_update=True, readout_format=".3f"
)
w_x = FloatSlider(
    value=0, min=0, max=0.277, step=0.001,
    description="X", continuous_update=True, readout_format=".3f"
)
w_new = Button(icon="plus", button_style="success",layout=widgets.Layout(width="40px"))
w_delete = Button(icon="trash", button_style="danger",layout=widgets.Layout(width="40px"))

class State:
    pattern = [[0, 0]]
    do_update=True

state = State()

for w in (w_f, w_x, w_y):
     w.layout.width = f"{fig_width * 80 - 50}px"
    
w_idx.layout.width = f"{fig_width * 80 - 160}px"

out = Output()
def update_plot(change=None):

    if not state.do_update:
        return

    fc = w_f.value
    pattern = np.array(state.pattern).T

    idx = max(0, min(int(w_idx.value), len(state.pattern)-1))

    patternX, patternY = pattern
    pattern = pattern.transpose()

    k = 2 * np.pi * fc / c

    xv, yv = np.meshgrid(x, y)
    thetav = np.atan2(yv, xv)
    rv = np.sqrt(xv*xv + yv*yv)
    
    p = np.stack((k * np.cos(thetav) * np.sin(rv), k * np.sin(thetav) * np.sin(rv)), axis=-1)

    def computeBeamPattern(micPositions):
        mp = micPositions
        return np.sum(np.array([np.exp(np.dot(p, mp[i])*1j) for i in range(len(mp))]), axis=0) / len(mp)

    def plotMesh(ax, z):
        polar_pts, cart_pts, xy = plot_points
        interp = RegularGridInterpolator(xy, np.abs(z), bounds_error=False)
        z = interp(cart_pts)
        c = ax.pcolormesh(*polar_pts, np.abs(z), shading='auto', cmap='viridis')
        plt.colorbar(c, ax=ax, label='Gain (linear)')

    z0 = computeBeamPattern(pattern)

    with out:
        clear_output(wait=True)

        fig = plt.figure(figsize=(fig_width, fig_width * 0.55))
        ax1 = fig.add_subplot(121)
        ax2 = fig.add_subplot(122, projection='polar')
        
        rect1 = Rectangle((0,0), 0.277, 0.32, color='gray', fc = 'none',lw = 2)
        rect2 = Rectangle((0.03,0.03), 0.21, 0.26, color='gray', fc = 'none',lw = 1)
        ax1.add_patch(rect1)
        ax1.add_patch(rect2)
        ax1.scatter(patternX[idx], patternY[idx], s=80, c='C1')
        ax1.scatter(patternX, patternY, s=20, c='C0')
        ax1.axis('equal')

        plotMesh(ax2, z0)
        ax2.set_title('Delay and sum beam pattern')

        plt.tight_layout()
        plt.show()

def add_mic(change=None):
    state.pattern.append([w_x.value, w_y.value])
    update_plot()

def delete_mic(change=None):
    idx = max(0, min(int(w_idx.value), len(state.pattern)-1))
    state.pattern.pop(idx)
    update_plot()

def set_mic_position(change=None):
    idx = max(0, min(int(w_idx.value), len(state.pattern)-1))
    state.pattern[idx] = [w_x.value, w_y.value]
    update_plot()

def set_idx(change=None):
    idx = max(0, min(int(w_idx.value), len(state.pattern)-1))
    x, y = state.pattern[idx]
    state.do_update = False
    w_x.value = x
    w_y.value = y
    state.do_update = True
    update_plot()

def set_array(array):
     state.pattern = array
     update_plot()

def get_array():
     return state.pattern

w_new.on_click(add_mic)
w_delete.on_click(delete_mic)
w_x.observe(set_mic_position, names="value")
w_y.observe(set_mic_position, names="value")
w_idx.observe(set_idx, names="value")

for w in (w_f, w_idx, w_x, w_y):
    w.observe(update_plot, names="value")

afw = make_array_file_widgets(get_array, set_array)

display(VBox([
    afw,
    w_f,
    HBox([w_idx, w_new, w_delete]),
    w_x,
    w_y,
    out
]))


update_plot(None)